# Equity Duration Notebook

This notebook constructs several **duration** measures for STOXX Europe 600 constituents:

1. **Operating cash-flow timing duration** (undiscounted)
2. **Discounted cash-flow duration** with fixed discount rate \(r = 12.5\%\)
3. **Discounted cash-flow duration** with firm-specific **CAPM cost of equity**
4. **Empirical (interest-rate sensitivity) duration** using daily changes in **2Y EUR OIS** (RIC `EUREON2Y=`)

> **Note on cash flows:** `CFPSMean_*` in this project is *cash flow from operations per share* (CFO per share), not net payouts to equity holders.


## 0. Setup

- Requires: `lseg.data` (LSEG Data Library), `pandas`, `numpy`
- You may need to authenticate / configure LSEG before running.


In [1]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## 1. Load the STOXX 600 constituent list

Expected columns (names can vary):
- RIC
- Company name
- Country of headquarters
- Market cap

Adjust `INPUT_XLSX` if needed.


In [21]:
ld.open_session()

date = "20001231"
cons = get_constituents_asof(date)

rics = cons["ConstituentRIC"].dropna().unique().tolist()

firm_parts = []
for rchunk in chunked(rics, n=200):
    firm_parts.append(enrich_firms_asof(rchunk, date))

firms = pd.concat(firm_parts, ignore_index=True)

panel_2000 = cons.merge(firms, on="ConstituentRIC", how="left")

ld.close_session()

print(panel_2000.head())
print("rows:", len(panel_2000), "unique rics:", panel_2000["ConstituentRIC"].nunique())
print("Missing CountryHQ:", panel_2000["CountryHQ"].isna().mean())
print("Missing MarketCap:", panel_2000["MarketCap"].isna().mean())

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


ValueError: Missing in firm enrichment: ['CountryHQ']. Got columns: ['ConstituentRIC', 'Country of Headquarters', 'MarketCap']

In [ ]:
# --- User input ---
INPUT_XLSX = DATA_DIR / "STOXX600Constituents202601.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())
mcap_col = next(c for c in df_const.columns if "MARKET" in c.upper() and "CAP" in c.upper())

df_basic = df_const[[ric_col, name_col, country_col, mcap_col]].copy()
df_basic.columns = ["RIC", "CompanyName", "CountryHQ", "MarketCap"]

df_basic = (
    df_basic
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

print(df_basic.head())
print("Number of firms:", len(df_basic))


       RIC         CompanyName               CountryHQ     MarketCap
0    A2.MI             A2A SpA                 Italien   7610.187355
1    AAF.L   Airtel Africa PLC  Vereinigtes Königreich  15313.262664
2   AAK.ST       AAK AB (publ)                Schweden   6142.691574
3    AAL.L  Anglo American PLC  Vereinigtes Königreich  42523.693525
4  AALB.AS         Aalberts NV             Niederlande   3222.012813
Number of firms: 600


## 2. Helper functions (LSEG pulls + finance math)

We define small helper functions for:
- Year-end price
- ROE (FY0)
- 5-year beta
- CAPM cost of equity
- Duration measures (discounted + undiscounted)


In [3]:
def get_year_end_price(ric: str, year: int = 2025, field: str = "TR.PriceClose") -> float:
    """Get the last available daily price in a given calendar year."""
    hist = ld.get_history(
        universe=[ric],
        fields=[field],
        start=f"{year}-01-01",
        end=f"{year}-12-31",
        interval="daily"
    )
    if hist is None or hist.empty:
        return np.nan

    # hist typically has Date as index and one column for the field
    df = hist.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        date_col = next(c for c in df.columns if "date" in str(c).lower())
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)

    # take last non-missing value
    s = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    return float(s.iloc[-1]) if len(s) else np.nan


def get_roe_fy0(ric: str) -> float:
    """ROE (FY0) from LSEG. Returns decimal (0.15 = 15%)."""
    roe_df = ld.get_data(
        universe=[ric],
        fields=["TR.F.ReturnAvgTotEqPct(period=FY0)"]
    )
    if roe_df is None or roe_df.empty:
        return np.nan

    roe_col = next((c for c in roe_df.columns if "Return on Average Total Equity" in c), None)
    if roe_col is None:
        # fallback: take the only non-id column
        roe_col = roe_df.columns[-1]

    roe = roe_df[roe_col].iloc[0]
    if roe is None:
        return np.nan

    return float(roe) / 100.0


def get_beta_5y(ric: str) -> float:
    """5Y beta from LSEG."""
    beta_df = ld.get_data(universe=[ric], fields=["TR.BetaFiveYear"])
    if beta_df is None or beta_df.empty:
        return np.nan

    beta_col = next((c for c in beta_df.columns if "Beta" in c), None)
    if beta_col is None:
        beta_col = beta_df.columns[-1]

    b = beta_df[beta_col].iloc[0]
    return float(b) if b is not None else np.nan


def coe_capm(beta: float, rf: float, erp: float) -> float:
    """CAPM cost of equity in decimals."""
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


def duration_discounted(CF: np.ndarray, P: float, r: float) -> float:
    """Implied duration with terminal value inferred from price."""
    if not np.isfinite(P) or P <= 0 or not np.isfinite(r) or r <= 0:
        return np.nan
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)
    return float(term1 + term2)


def duration_undiscounted(CF: np.ndarray) -> float:
    """Cash-flow timing duration (undiscounted)."""
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan
    t = np.arange(1, T + 1, dtype=float)
    return float((t * CF).sum() / CF.sum()) if CF.sum() != 0 else np.nan


## 3. Pull firm-level inputs (price, ROE, beta) and compute CAPM \(r_i\)

- **Price (Dec 31, 2025):** last available daily close in 2025
- **ROE FY0:** used for descriptive stats (not for duration construction here)
- **Beta (5Y):** used for CAPM \(r_i\)


In [ ]:
# --- User choices for CAPM ---
RF = 0.025   # 2.5% risk-free (decimal)
ERP = 0.05   # 5% equity risk premium (decimal)

ld.open_session()

# Year-end price
prices = []
for ric in df_basic["RIC"]:
    try:
        prices.append(get_year_end_price(ric, year=2025, field="TR.PriceClose"))
    except Exception:
        prices.append(np.nan)
df_basic["Price_2025_12_31"] = prices

# ROE FY0
roe_vals = []
for ric in df_basic["RIC"]:
    try:
        roe_vals.append(get_roe_fy0(ric))
    except Exception:
        roe_vals.append(np.nan)
df_basic["ROE_FY0"] = roe_vals

# Beta + CAPM CoE
betas = []
coes = []
for ric in df_basic["RIC"]:
    try:
        b = get_beta_5y(ric)
    except Exception:
        b = np.nan
    betas.append(b)
    coes.append(coe_capm(b, rf=RF, erp=ERP))

df_basic["BETA_5Y"] = betas
df_basic["COE_CAPM"] = coes

ld.close_session()

df_basic[["RIC","Price_2025_12_31","ROE_FY0","BETA_5Y","COE_CAPM"]].head()


In [26]:
# Vorbereitung zum Speichern als Parquet

def make_arrow_safe(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()

    for c in df2.columns:
        # datetime ok
        if pd.api.types.is_datetime64_any_dtype(df2[c]):
            continue

        # Pandas nullable integers/booleans -> float/bool/object
        if str(df2[c].dtype) in ["Int64", "Int32", "Int16", "Int8"]:
            df2[c] = df2[c].astype("float64")
        elif str(df2[c].dtype) in ["boolean"]:
            df2[c] = df2[c].astype("object")

        # pandas string dtype -> object (safe)
        elif pd.api.types.is_string_dtype(df2[c]) and str(df2[c].dtype).startswith("string"):
            df2[c] = df2[c].astype("object")

        # mixed objects: keep, but ensure pandas.NA -> np.nan
        if df2[c].dtype == "object":
            df2[c] = df2[c].where(df2[c].notna(), None)

    return df2

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df_safe = make_arrow_safe(df)
    df_safe.to_parquet(path, index=False, engine="pyarrow")
    print(f"Saved: {path}")

# Tabelle in intermediate Speichern

firm_cols = [
    "RIC", "CompanyName", "CountryHQ",
    "MarketCap", "Price_2025_12_31",
    "ROE_FY0", "BETA_5Y", "COE_CAPM"
]

df_firm_inputs = df_basic[firm_cols].copy()

save_parquet(df_firm_inputs, "firm_inputs")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/firm_inputs.parquet


## 4. Analyst operating cash-flow forecasts (CFPS)

This notebook expects these columns already in `df_basic`:

- `CFPSMean_FY25` … `CFPSMean_FY29`


In [35]:
years = [2025, 2026, 2027, 2028, 2029]
rics = df_basic["RIC"].dropna().astype(str).unique().tolist()

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

ld.open_session()

try:
    for y in years:
        col_name = f"CFPSMean_FY{str(y)[-2:]}"
        if col_name not in df_basic.columns:
            df_basic[col_name] = np.nan

        # Sammle alle Batch-Ergebnisse in einem Mapping (RIC -> value)
        mapping = {}

        for batch in chunks(rics, 200):
            df_y = ld.get_data(
                universe=batch,
                fields=["TR.CFPSMean"],
                parameters={"SDate": f"{y}-12-01", "EDate": f"{y}-12-01", "FRQ": "FY"}
            )

            if df_y is None or df_y.empty:
                continue

            # Stabilisiert dtypes (hilft gegen LSEG/pandas downcasting warnings)
            df_y = df_y.infer_objects(copy=False)

            # Identifier-Spalte robust finden (manchmal "Instrument")
            id_col = "Instrument" if "Instrument" in df_y.columns else df_y.columns[0]

            # CFPS-Spalte robust finden
            cfps_col = next(
                (c for c in df_y.columns
                 if ("Cash Flow Per Share" in str(c)) and ("Mean" in str(c))),
                None
            )
            if cfps_col is None:
                # Fallback: wenn LSEG dir direkt "TR.CFPSMean" als Spaltennamen gibt
                cfps_col = next((c for c in df_y.columns if "CFPS" in str(c).upper()), None)

            if cfps_col is None:
                continue

            tmp = df_y[[id_col, cfps_col]].copy()
            tmp = tmp.rename(columns={id_col: "RIC", cfps_col: col_name})
            tmp["RIC"] = tmp["RIC"].astype(str)
            tmp[col_name] = pd.to_numeric(tmp[col_name], errors="coerce")

            # Update mapping (letzter gültiger Wert gewinnt)
            tmp = tmp.dropna(subset=[col_name])
            mapping.update(dict(zip(tmp["RIC"], tmp[col_name])))

        # In df_basic schreiben (ohne merges im Loop)
        s_new = df_basic["RIC"].map(mapping)

        # falls schon Werte da sind: keep existing, fill missing with new
        df_basic[col_name] = df_basic[col_name].combine_first(s_new)

finally:
    ld.close_session()

print(df_basic[["RIC", "CompanyName", "CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]].head())
print("Missing FY25:", df_basic["CFPSMean_FY25"].isna().sum())

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit

       RIC         CompanyName  CFPSMean_FY25  CFPSMean_FY26  CFPSMean_FY27  \
0    A2.MI             A2A SpA        0.57333        0.53667        0.57667   
1    AAF.L   Airtel Africa PLC          0.395          0.469         0.6565   
2   AAK.ST       AAK AB (publ)          10.58          16.16       17.64333   
3    AAL.L  Anglo American PLC        3.65286          5.178        5.57833   
4  AALB.AS         Aalberts NV           4.07           4.68           4.95   

   CFPSMean_FY28  CFPSMean_FY29  
0           0.58           0.57  
1            0.7           0.81  
2           20.7           <NA>  
3            5.5           3.68  
4           <NA>           <NA>  
Missing FY25: 95


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [14]:
cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

# Ensure numeric
for c in cf_cols:
    df_basic[c] = pd.to_numeric(df_basic[c], errors="coerce")

df_basic[["RIC"] + cf_cols].head()

cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

df_cfps = df_basic[["RIC"] + cf_cols].copy()
save_parquet(df_cfps, "cfps_forecasts")


Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/cfps_forecasts.parquet


## 5. Duration measures from CFPS

We compute three timing measures:

- `Duration_r125`: discounted duration with fixed \(r=12.5\%\)
- `Duration_CAPM`: discounted duration with firm-specific \(r_i\) from CAPM
- `Duration_undiscounted`: pure timing (no discounting)

All durations use **price per share** and **CFPS** (per share), so units are consistent.


In [ ]:
R_FIXED = 0.125

# Build CF arrays per row
def row_cf_array(row) -> np.ndarray:
    s = pd.to_numeric(pd.Series([row.get(c) for c in cf_cols]), errors="coerce")
    return s.to_numpy(dtype=float)

df_basic["Duration_r125"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=R_FIXED),
    axis=1
)

df_basic["Duration_CAPM"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=row["COE_CAPM"]),
    axis=1
)

df_basic["Duration_undiscounted"] = df_basic.apply(
    lambda row: duration_undiscounted(row_cf_array(row)),
    axis=1
)

df_basic[["RIC","Duration_r125","Duration_CAPM","Duration_undiscounted"]].head()


,RIC,Duration_r125,Duration_CAPM,Duration_undiscounted
0,A2.MI,4.200683,2.723758,3.012927
1,AAF.L,13.936659,15.422047,3.350107
2,AAK.ST,11.129164,18.408584,2.744635
3,AAL.L,13.939346,14.514973,3.015951
4,AALB.AS,8.148887,9.672361,2.064234


## 6. Empirical duration: interest-rate sensitivities

We estimate firm-specific sensitivities to **daily changes in the 2Y EUR OIS** rate:

$
r_{i,t} = a_i + b_i \Delta y_t + \varepsilon_{i,t}
\quad\Rightarrow\quad
D^{emp}_i = -b_i
$

### 6.1 Pull the daily 2Y OIS series (RIC `EUREON2Y=`)

We use the field that is available in your setup: `TR.MIDPRICE` (rate in **percent**).


In [21]:
ld.open_session()

# 2Y EUR OIS / EONIA-style series
ric_rate = "EUREON2Y="
field_rate = "TR.MIDPRICE"

y_df = ld.get_history(
    universe=[ric_rate],
    fields=[field_rate],
    start="2015-01-01",
    end="2025-12-31",
    interval="daily"
)

ld.close_session()

# Build daily rate changes (in percentage points)
rate_df = y_df.copy().reset_index()
rate_df.columns = ["date", "y"]   # y in percent
rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")
rate_df = rate_df.dropna(subset=["y"]).sort_values("date")
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date","dy"]].dropna()

rates_daily.head()


,date,dy
1,2015-01-02,-0.012
2,2015-01-05,0.004
3,2015-01-06,-0.001
4,2015-01-07,0.001
5,2015-01-08,0.01


In [22]:
df_2yOIS = rates_daily.copy()
save_parquet(df_2yOIS, "rates_2yOIS_daily")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/rates_2yOIS_daily.parquet


### 6.2 Pull daily stock prices and compute daily returns

We retrieve daily prices for all RICs, compute simple returns, then merge with `rates_daily`.


In [24]:
def chunk_list(x, n=40):
    for i in range(0, len(x), n):
        yield x[i:i+n]


def get_prices_daily(rics, start="2015-01-01", end="2025-12-31",
                     field="TR.PriceClose", batch_size=40):
    """Fast pull of daily prices (one field) in batches; returns long format."""
    out = []
    for batch in chunk_list(list(rics), n=batch_size):
        df = ld.get_history(
            universe=batch,
            fields=[field],
            start=start,
            end=end,
            interval="daily"
        )
        if df is None or df.empty:
            continue

        df_w = df.copy()
        if not isinstance(df_w.index, pd.DatetimeIndex):
            df_w = df_w.reset_index()
            date_col = next((c for c in df_w.columns if "date" in str(c).lower()), None)
            df_w[date_col] = pd.to_datetime(df_w[date_col])
            df_w = df_w.set_index(date_col)

        df_long = df_w.stack(dropna=False).reset_index()
        df_long.columns = ["date", "RIC", "price"]
        df_long["date"] = pd.to_datetime(df_long["date"])
        df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")
        df_long = df_long.dropna(subset=["price"])
        out.append(df_long)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["date","RIC","price"])


ld.open_session()
prices_daily = get_prices_daily(df_basic["RIC"].tolist(), start="2015-01-01", end="2025-12-31",
                                field="TR.PriceClose", batch_size=40)
ld.close_session()

# daily returns
prices_daily = prices_daily.sort_values(["RIC","date"])
prices_daily["ret"] = prices_daily.groupby("RIC")["price"].pct_change()
rets_daily = prices_daily.dropna(subset=["ret"])[["date","RIC","ret"]].copy()

rets_daily.head()

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_31319/1757957386.py:28:FutureWarning: The previous implemen

,date,RIC,ret
28,2015-01-05,A2.MI,-0.032914
58,2015-01-06,A2.MI,-0.014851
84,2015-01-07,A2.MI,-0.005653
114,2015-01-08,A2.MI,0.044852
144,2015-01-09,A2.MI,-0.020556


In [27]:
# Sortieren & Index bereinigen (Best Practice)
rets_daily = (
    rets_daily
    .sort_values(["date", "RIC"])
    .reset_index(drop=True)
)

# Als Parquet in Project_Data/intermediate speichern
save_parquet(rets_daily, "returns_daily")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/returns_daily.parquet


### 6.3 Estimate firm-level rate betas and empirical duration

In [28]:
def estimate_rate_beta_duration(rets_daily: pd.DataFrame, rates_daily: pd.DataFrame, min_obs: int = 200) -> pd.DataFrame:
    """Estimate r_{i,t} = a_i + b_i dy_t; return b_i and D_emp=-b_i."""
    df = rets_daily.merge(rates_daily, on="date", how="inner").dropna(subset=["ret","dy"]).copy()

    out = []
    for ric, g in df.groupby("RIC"):
        if len(g) < min_obs:
            continue
        x = g["dy"].to_numpy(float)   # dy in percentage points
        y = g["ret"].to_numpy(float)

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]
        out.append({"RIC": ric, "beta_rate_2yOIS": b, "D_emp_2yOIS": -b, "n_obs": len(g)})

    return pd.DataFrame(out)

dur_emp = estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200)
df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC","D_emp_2yOIS","beta_rate_2yOIS","n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())
print(df_basic[["Duration_r125","D_emp_2yOIS"]].corr(method="spearman"))
print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())


       RIC  D_emp_2yOIS  beta_rate_2yOIS   n_obs
0    A2.MI     0.040532        -0.040532  2792.0
1    AAF.L    -0.038529         0.038529  1644.0
2   AAK.ST     0.002520        -0.002520  2762.0
3    AAL.L    -0.026897         0.026897  2776.0
4  AALB.AS    -0.026285         0.026285  2813.0
count    596.000000
mean      -0.009672
std        0.037235
min       -0.156976
25%       -0.029717
50%       -0.007501
75%        0.012449
max        0.108576
Name: D_emp_2yOIS, dtype: float64
               Duration_r125  D_emp_2yOIS
Duration_r125       1.000000     0.182381
D_emp_2yOIS         0.182381     1.000000
Share with D_emp: 0.9933333333333333


## 7. Net-Payout-Based Duration

In [67]:
import time, random
import numpy as np
import pandas as pd
import lseg.data as ld
from lseg.data.errors import LDError

rics = df_basic["RIC"].dropna().unique().tolist()

FIELDS = {
    "MKT_CAP":         "TR.CompanyMarketCap",
    "NET_PAYOUT_CORE": "TR.F.CashDivPaidComStockBuybackNet",
    "EQUITY_ISSUANCE": "TR.F.StockTotIssuanceRetNetCF",
    "BETA_5Y":         "TR.BetaFiveYear",
}

def parse_ld_number(x):
    if x is None or pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in {"<NA>", "NA", "N/A", ""}:
        return np.nan
    is_paren_neg = s.startswith("(") and s.endswith(")")
    s = s.replace(",", "").replace("(", "").replace(")", "")
    val = pd.to_numeric(s, errors="coerce")
    if pd.isna(val):
        return np.nan
    return -val if is_paren_neg else val

def is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return ("too many requests" in msg) or ("rate" in msg and "limit" in msg)

def get_data_with_retry(universe, fields, parameters, max_retries=6):
    attempt = 0
    while True:
        try:
            return ld.get_data(universe=universe, fields=fields, parameters=parameters)
        except LDError as e:
            attempt += 1
            if attempt > max_retries or not is_rate_limit_error(e):
                raise
            time.sleep((2 ** attempt) + random.random())  # backoff + jitter

def pull_fy_multifield(universe, field_map, year, mmdd="12-31"):
    sdate = f"{year}-{mmdd}"
    edate = f"{year}-{mmdd}"
    req_fields = list(field_map.values())

    df = get_data_with_retry(
        universe=universe,
        fields=req_fields,
        parameters={"SDate": sdate, "EDate": edate, "FRQ": "FY"},
        max_retries=8
    )

    if df is None or df.empty:
        return pd.DataFrame(columns=["RIC", "Year"] + list(field_map.keys()))

    df = df.rename(columns={"Instrument": "RIC"})
    df["Year"] = year

    out = df[["RIC", "Year"]].copy()

    # map aliases to returned columns (handles uppercase / label variations)
    for alias, code in field_map.items():
        col = None
        if code in df.columns:
            col = code
        else:
            codeU = code.upper()
            col = next((c for c in df.columns if str(c).upper() == codeU), None) \
                  or next((c for c in df.columns if codeU in str(c).upper()), None)

        out[alias] = df[col].apply(parse_ld_number) if col else np.nan

    return out

In [68]:
ld.open_session()

test_rics = [r for r in ["SAPG.DE", "SIEGn.DE", "VOWG_p.DE", "RIO.L", "ULVR.L"] if r in rics]
if len(test_rics) == 0:
    test_rics = rics[:5]

df_test = pull_fy_multifield(test_rics, FIELDS, year=2024, mmdd="12-31")

ld.close_session()

display(df_test)
print("Coverage (test) NET_PAYOUT_CORE:", df_test["NET_PAYOUT_CORE"].notna().mean())
print("Coverage (test) EQUITY_ISSUANCE:", df_test["EQUITY_ISSUANCE"].notna().mean())

LDError: Too many requests, please try again later. Requested universes: ['SAPG.DE', 'SIEGn.DE', 'VOWG_p.DE', 'RIO.L', 'ULVR.L']. Requested fields: ['TR.COMPANYMARKETCAP', 'TR.F.CASHDIVPAIDCOMSTOCKBUYBACKNET', 'TR.F.STOCKTOTISSUANCERETNETCF', 'TR.BETAFIVEYEAR']

## 8. Final results table (rounded)

We include:
- Firm descriptors
- CAPM inputs
- CFPS forecasts
- All duration measures (rounded to 2 decimals)


In [33]:
cols = [
    "RIC","CompanyName","CountryHQ","MarketCap","Price_2025_12_31",
    "ROE_FY0","BETA_5Y","COE_CAPM",
    "CFPSMean_FY25","CFPSMean_FY26","CFPSMean_FY27","CFPSMean_FY28","CFPSMean_FY29",
    "Duration_r125","Duration_CAPM","Duration_undiscounted","D_emp_2yOIS"
]

df_results = df_basic[cols].copy()

# percentages for presentation
df_results["ROE_FY0"] = df_results["ROE_FY0"] * 100
df_results["COE_CAPM"] = df_results["COE_CAPM"] * 100

df_results = df_results.rename(columns={
    "Price_2025_12_31": "Price (Dec 31, 2025)",
    "ROE_FY0": "ROE (%)",
    "BETA_5Y": "Beta (5Y)",
    "COE_CAPM": "Cost of Equity CAPM (%)",
    "CFPSMean_FY25": "CFPS FY25",
    "CFPSMean_FY26": "CFPS FY26",
    "CFPSMean_FY27": "CFPS FY27",
    "CFPSMean_FY28": "CFPS FY28",
    "CFPSMean_FY29": "CFPS FY29",
    "Duration_r125": "Duration (r = 12.5%)",
    "Duration_CAPM": "Duration (CAPM r)",
    "Duration_undiscounted": "Duration (undiscounted)",
    "D_emp_2yOIS": "Empirical Duration (2Y OIS)"
})

df_results.to_excel(TABLE_DIR / "final_results_duration.xlsx", index=False)

df_results_rounded = df_results.copy()

# round most numeric columns
df_results_rounded = df_results_rounded.round({
    "MarketCap": 2,
    "Price (Dec 31, 2025)": 2,
    "ROE (%)": 2,
    "Beta (5Y)": 2,
    "Cost of Equity CAPM (%)": 2,
    "CFPS FY25": 2, "CFPS FY26": 2, "CFPS FY27": 2, "CFPS FY28": 2, "CFPS FY29": 2,
})

# round all duration columns automatically
duration_cols = [c for c in df_results_rounded.columns if "Duration" in c]
df_results_rounded[duration_cols] = df_results_rounded[duration_cols].round(2)

df_results_rounded.sort_values("MarketCap", ascending=False).head(20)


,RIC,CompanyName,CountryHQ,MarketCap,"Price (Dec 31, 2025)",ROE (%),Beta (5Y),Cost of Equity CAPM (%),CFPS FY25,CFPS FY26,CFPS FY27,CFPS FY28,CFPS FY29,Duration (r = 12.5%),Duration (CAPM r),Duration (undiscounted),Empirical Duration (2Y OIS)
57,ASML.AS,ASML Holding NV,Niederlande,393698.46,921.40,43.68,2.12,13.08,22.02,27.06,36.4,47.21,52.61,12.54,12.25,3.44,-0.00
343,LVMH.PA,LVMH Moet Hennessy Louis Vuitton SE,Frankreich,316187.84,645.00,19.64,1.42,9.58,33.91,35.26,37.39,47.19,<NA>,11.15,13.00,2.64,-0.01
440,ROG.S,Roche Holding AG,Schweiz,296035.54,328.20,26.47,1.03,7.65,24.48,25.44,27.21,28.74,29.67,10.76,13.75,3.10,0.01
382,NOVN.S,Novartis AG,Schweiz,258020.21,109.60,26.28,0.79,6.43,9.26,9.45,10.07,11.69,14.57,10.17,13.95,3.23,-0.01
457,SAPG.DE,SAP SE,Deutschland,254011.86,208.35,7.06,1.03,7.67,7.54,9.01,10.69,12.27,15.42,12.04,15.76,3.35,0.01
70,AZN.L,AstraZeneca PLC,Vereinigtes Königreich,253387.46,13790.00,17.59,0.87,6.83,9.8,10.76,12.11,12.55,16.74,13.97,20.58,3.25,0.00
264,HSBA.L,HSBC Holdings PLC,Vereinigtes Königreich,236625.46,1173.80,12.99,1.24,8.68,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,-0.07
263,HRMS.PA,Hermes International SCA,Frankreich,225891.51,2122.00,28.47,1.40,9.50,43.96,49.3,59.58,70.23,<NA>,12.19,14.46,2.70,0.01
371,NESN.S,Nestle SA,Schweiz,205994.10,78.74,30.58,0.94,7.18,5.63,5.74,6.08,6.76,<NA>,10.58,14.68,2.58,0.01
487,SIEGn.DE,Siemens Aktiengesellschaft,Deutschland,201698.65,239.15,13.37,1.36,9.32,13.96,15.32,17.29,38.42,<NA>,10.41,12.19,2.94,-0.03
